# 06 — LGD Model Training

This notebook trains six regression models for Loss Given Default (LGD) estimation. Each model is tuned via RandomizedSearchCV on a random sample of 100,000 observations (from the ~128k defaulted loans) and evaluated on a held-out test set. The best model (CatBoost, RMSE = 22.90, R² = 0.6651) is selected for portfolio inference.

**Input:** `data/processed/checkpoint_df_LGD_training.pkl`, `data/processed/features_LGD_num_for_training.pkl`, `data/processed/features_LGD_cat_for_training.pkl`  
**Output:** `models/catboost_LGD.pkl`, `models/LGD_model_comparison.csv`

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import shap
import os

from sklearn.model_selection import train_test_split

from train_models_LGD import train_all_models_LGD
from pipeline_LGD     import FeaturePipeline_LGD
from preprocess       import preprocess_features_LGD

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_df_LGD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_LGD_clean = data["df_LGD_clean"]

with open("data/processed/features_LGD_num_for_training.pkl", "rb") as f:
    num_features_LGD = pickle.load(f)

with open("data/processed/features_LGD_cat_for_training.pkl", "rb") as f:
    cat_features_LGD = pickle.load(f)

features_LGD = num_features_LGD + cat_features_LGD

print(f"df_LGD_clean shape: {df_LGD_clean.shape}")
print(f"Features: {features_LGD}")

## 2. Train / Test Split

Random split (no stratification — regression target is continuous).

In [ ]:
X = df_LGD_clean[features_LGD]
y = df_LGD_clean["Target_Variable_Loss"]

X_train_LGD, X_test_LGD, y_train_LGD, y_test_LGD = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Train: {X_train_LGD.shape} | Test: {X_test_LGD.shape}")
print(f"Mean LGD train: {y_train_LGD.mean():.4f} | test: {y_test_LGD.mean():.4f}")

## 3. Preprocessing

RobustScaler fitted on LGD train only (defaulted population), applied to both train and test. One-Hot Encoding for `loan_term_months`.

In [ ]:
X_train_LGD, X_test_LGD, scaler_LGD = preprocess_features_LGD(
    X_train_LGD, X_test_LGD, cat_features_LGD, num_features_LGD
)

print(f"X_train_LGD shape: {X_train_LGD.shape} | X_test_LGD shape: {X_test_LGD.shape}")

## 4. Model Training

Six regression models are trained:
- Ridge Regression (interpretable baseline with L2 regularization)
- Decision Tree
- Random Forest
- XGBoost
- LightGBM
- CatBoost

All models tuned via RandomizedSearchCV (100k sample, 5-fold CV, optimizing RMSE). Results saved to `models/`.

In [ ]:
results_LGD = train_all_models_LGD(
    X_train_LGD, y_train_LGD,
    X_test_LGD,  y_test_LGD,
    sample_size=100_000
)

In [ ]:
# Results summary — sorted by RMSE ascending
results_LGD

## 5. SHAP Analysis

In [ ]:
# Load best model
with open("models/catboost_LGD.pkl", "rb") as f:
    model_LGD = pickle.load(f)

# SHAP on a representative sample
X_shap_LGD = pd.concat([X_train_LGD, X_test_LGD], axis=0).sample(n=20_000, random_state=42)

explainer_LGD   = shap.TreeExplainer(model_LGD)
shap_values_LGD = explainer_LGD.shap_values(X_shap_LGD)

if isinstance(shap_values_LGD, list):
    shap_values_LGD = shap_values_LGD[1]

print(f"SHAP values shape: {shap_values_LGD.shape}")

shap.summary_plot(shap_values_LGD, X_shap_LGD, max_display=X_shap_LGD.shape[1])

## 6. LGD Inference on Full Portfolio

The LGD model is applied to the full portfolio — including non-defaulted loans — using the pipeline parameters fitted on the defaulted training population. See `docs/inference_strategy.md` for full justification of this approach and its known limitations.

In [ ]:
# Load df_master_features with PD already computed
with open("data/processed/checkpoint_df_master_with_PD.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

In [ ]:
# Fit LGD pipeline on defaulted training population
pipeline_LGD = FeaturePipeline_LGD()
pipeline_LGD.fit(
    df_master_features.loc[
        df_master_features["Target_Variable_Default"] == 1,
        features_LGD
    ]
)

# Transform full dataset with LGD pipeline parameters
X_full_LGD = pipeline_LGD.transform(df_master_features[features_LGD])

# Inference
y_pred_LGD = model_LGD.predict(X_full_LGD)
df_master_features["LGD"] = y_pred_LGD

print(f"LGD computed for {len(y_pred_LGD):,} loans")
print(f"Mean LGD   : {y_pred_LGD.mean():.4f}")
print(f"Median LGD : {np.median(y_pred_LGD):.4f}")

In [ ]:
# Calibration check — compare predicted vs observed LGD on defaulted population
mask_default = df_master_features["Target_Variable_Default"] == 1

print(f"Mean LGD predicted (defaulted) : {df_master_features.loc[mask_default, 'LGD'].mean():.4f}")
print(f"Mean LGD real (defaulted)      : {df_master_features.loc[mask_default, 'Target_Variable_Loss'].mean():.4f}")
print(f"Median LGD predicted (defaulted): {df_master_features.loc[mask_default, 'LGD'].median():.4f}")
print(f"Median LGD real (defaulted)     : {df_master_features.loc[mask_default, 'Target_Variable_Loss'].median():.4f}")

In [ ]:
# Save pipeline and df_master_features with PD and LGD columns
pipeline_LGD.save("models/pipeline_LGD.pkl")

with open("data/processed/checkpoint_df_master_with_PD_LGD.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

print("Saved pipeline and df_master_features with PD and LGD columns.")